In [2]:
# https://grok.com/share/bGVnYWN5_654f9fe9-1e40-445e-ba48-6e86d68645d6

import pandas as pd
import datetime
from pandasql import sqldf
import sqlite3
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # Or another classifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder  # Use OneHotEncoder




pd.options.display.max_rows = 50
pd.options.display.max_columns = 100

mysqldf = lambda q: sqldf(q, globals())

conn = sqlite3.connect('optionsQuotes1.db')
#c = conn.cursor()
df = pd.read_sql('select * from data',conn)
conn.close()
df['time_converted'] = pd.to_datetime(df['time_converted'])

In [3]:
df.head(100)

,index,volume_options,volume_weighted_options,open_options,close_options,high_options,low_options,timestamp_options,number of trades_options,ticker,full_name,type,strike,expiry,equity_start_price,time_converted,open_equity,close_equity,high_equity,low_equity,timestamp_equity,number of trades_equity,_merge,equity_pct_change,options_pct_change,options_earliest_open,file_source,equity_pct_change_normalized,day_classification,DTE,day_name,DTE_adjusted
0,0,4,10.8500,11.53,10.45,11.53,10.45,1672757100000,4,SPY,O:SPY230103C00373000,call,373,2023-01-03 00:00:00,385.29,2023-01-03 09:45:00,385.2900,381.7100,385.40,381.6601,1.672757e+12,35250.0,both,1.000000,1.000000,11.53,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,0.000000,high,0,Tuesday,0
1,1,14,8.9964,9.25,8.50,9.25,8.50,1672758000000,6,SPY,O:SPY230103C00373000,call,373,2023-01-03 00:00:00,385.29,2023-01-03 10:00:00,381.7700,381.1650,382.58,380.9300,1.672758e+12,27992.0,both,0.990864,0.802255,11.53,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-0.913598,high,0,Tuesday,0
2,2,41,7.3554,7.31,7.82,7.82,7.29,1672758900000,5,SPY,O:SPY230103C00373000,call,373,2023-01-03 00:00:00,385.29,2023-01-03 10:15:00,381.1800,380.9400,381.37,380.1450,1.672759e+12,24872.0,both,0.989333,0.633998,11.53,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.066729,high,0,Tuesday,0
3,3,19,7.7484,7.44,7.90,7.90,7.44,1672759800000,6,SPY,O:SPY230103C00373000,call,373,2023-01-03 00:00:00,385.29,2023-01-03 10:30:00,380.8900,381.0600,381.39,380.0700,1.672760e+12,24431.0,both,0.988580,0.645273,11.53,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.141997,high,0,Tuesday,0
4,4,12,6.8425,6.73,6.89,6.98,6.67,1672762500000,7,SPY,O:SPY230103C00373000,call,373,2023-01-03 00:00:00,385.29,2023-01-03 11:15:00,379.6600,380.0382,380.46,379.4800,1.672762e+12,25032.0,both,0.985388,0.583695,11.53,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.461237,high,0,Tuesday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,56,5.8621,7.20,5.11,7.30,5.11,1672761600000,9,SPY,O:SPY230103C00375000,call,375,2023-01-03 00:00:00,385.29,2023-01-03 11:00:00,382.0499,379.6521,382.33,379.4800,1.672762e+12,22782.0,both,0.991590,0.750782,9.59,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-0.840951,high,0,Tuesday,0
96,96,25,5.1360,5.07,5.25,5.25,5.04,1672762500000,3,SPY,O:SPY230103C00375000,call,375,2023-01-03 00:00:00,385.29,2023-01-03 11:15:00,379.6600,380.0382,380.46,379.4800,1.672762e+12,25032.0,both,0.985388,0.528676,9.59,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.461237,high,0,Tuesday,0
97,97,27,4.5571,4.50,4.28,4.70,4.28,1672763400000,9,SPY,O:SPY230103C00375000,call,375,2023-01-03 00:00:00,385.29,2023-01-03 11:30:00,380.0400,379.1000,380.05,378.9700,1.672763e+12,24003.0,both,0.986374,0.469239,9.59,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.362610,high,0,Tuesday,0
98,98,30,4.1037,4.15,4.19,4.19,3.99,1672764300000,8,SPY,O:SPY230103C00375000,call,375,2023-01-03 00:00:00,385.29,2023-01-03 11:45:00,379.1000,379.0700,379.34,378.6500,1.672764e+12,19463.0,both,0.983934,0.432742,9.59,0-2DTE_spy_options_01Jan23-28Feb23v2.pkl,-1.606582,high,0,Tuesday,0


In [11]:
# daily_summary  = mysqldf(""" 
#              SELECT
# DATE(Time_Converted) AS 'Date',
# MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS 'Call_Max_Daily',
# MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS 'Put_Max_Daily'
# FROM df
# WHERE DTE_ADJUSTED = 0 AND options_earliest_open>0.2
# GROUP BY DATE(Time_Converted)
#         """)

daily_summary  = mysqldf(""" 
    SELECT 
    o.Date,
    o.Call_Max_Daily,
    o.Put_Max_Daily,
    e.equity_open,
    e.close_equity,
    e.high_equity,
    e.low_equity
FROM (
    SELECT 
        DATE(Time_Converted) AS 'Date',
        MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS Call_Max_Daily,
        MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS Put_Max_Daily
    FROM df
    WHERE DTE_ADJUSTED = 0 
        AND options_earliest_open > 0.2
        AND ticker = 'SPY'
    GROUP BY DATE(Time_Converted)
) AS o
LEFT JOIN (
    SELECT
        daily_agg.Date,
        daily_agg.equity_open,
        daily_agg.high_equity,
        daily_agg.low_equity,
        last_candle.close_equity
    FROM (
        SELECT
            DATE(time_converted) AS Date,
            MAX(equity_start_price) AS equity_open,
            MAX(high_equity) AS high_equity,
            MIN(low_equity) AS low_equity
        FROM df
        WHERE ticker = 'SPY'
        GROUP BY DATE(time_converted)
    ) AS daily_agg
    LEFT JOIN (
        SELECT
            DATE(time_converted) AS Date,
            close_equity
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (PARTITION BY DATE(time_converted) ORDER BY TIME(time_converted) DESC) AS rn
            FROM df
            WHERE ticker = 'SPY'
                AND close_equity IS NOT NULL
                AND TIME(time_converted) <= '15:45:00'
        ) AS ranked_data
        WHERE rn = 1
    ) AS last_candle ON daily_agg.Date = last_candle.Date
) AS e ON o.Date = e.Date
        """)


daily_summary['Max'] = daily_summary[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary['Date'] = pd.to_datetime(daily_summary['Date'])
daily_summary['weekday'] = daily_summary['Date'].dt.day_name()


In [12]:
daily_summary

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,close_equity,high_equity,low_equity,Max,weekday
0,2023-01-03,1.000000,10.880000,385.290,380.8400,385.4000,377.8310,10.880000,Tuesday
1,2023-01-04,2.363636,1.419355,382.630,383.7400,385.8800,380.0000,2.363636,Wednesday
2,2023-01-05,1.411765,1.078755,379.710,379.4000,381.8400,378.7600,1.411765,Thursday
3,2023-01-06,13.225806,1.000000,380.210,388.0300,389.2500,379.4127,13.225806,Friday
4,2023-01-09,2.790698,1.705069,390.490,387.8657,393.7000,387.6700,2.790698,Monday
...,...,...,...,...,...,...,...,...,...
219,2023-11-22,1.069966,2.871795,456.140,455.0300,456.2200,453.8895,2.871795,Wednesday
220,2023-11-24,1.346939,1.042038,454.930,455.0000,455.6000,454.7300,1.346939,Friday
221,2023-11-27,1.612903,1.019108,454.360,454.5000,455.4901,454.0799,1.612903,Monday
222,2023-11-28,4.509804,1.215000,453.605,454.8800,456.2700,453.5100,4.509804,Tuesday


In [6]:
daily_summary

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,close_equity,Max,weekday
0,2023-01-03,1.000000,10.880000,385.290,380.8400,10.880000,Tuesday
1,2023-01-04,2.363636,1.419355,382.630,383.7400,2.363636,Wednesday
2,2023-01-05,1.411765,1.078755,379.710,379.4000,1.411765,Thursday
3,2023-01-06,13.225806,1.000000,380.210,388.0300,13.225806,Friday
4,2023-01-09,2.790698,1.705069,390.490,387.8657,2.790698,Monday
...,...,...,...,...,...,...,...
219,2023-11-22,1.069966,2.871795,456.140,455.0300,2.871795,Wednesday
220,2023-11-24,1.346939,1.042038,454.930,455.0000,1.346939,Friday
221,2023-11-27,1.612903,1.019108,454.360,454.5000,1.612903,Monday
222,2023-11-28,4.509804,1.215000,453.605,454.8800,4.509804,Tuesday


In [19]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

# Define hyperparameters for RandomForestClassifier
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Define classification ranges for day_classification
very_high_range = 4
high_range = 2
average_range = 1.5

def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

# Add initial features to daily_summary
daily_summary['spy_direction_lag1'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(1)
daily_summary['intraday_range_lag1'] = (daily_summary['high_equity'] - daily_summary['low_equity']).shift(1)
daily_summary['spy_direction_lag2'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(2)
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

def create_lags(df):
    df['d1'] = df.loc[:, 'day_classification'].shift(1)
    df['d2'] = df.loc[:, 'day_classification'].shift(2)
    df['d3'] = df.loc[:, 'day_classification'].shift(3)
    df['d4'] = df.loc[:, 'day_classification'].shift(4)
    df['d5'] = df.loc[:, 'day_classification'].shift(5)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day - 1) // 7 + 1)
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['max_lag1'] = df['Max'].shift(1)
    df['max_lag2'] = df['Max'].shift(2)
    df['max_lag3'] = df['Max'].shift(3)
    df['spy_direction_lag1'] = df['spy_direction_lag1']
    df['intraday_range_lag1'] = df['intraday_range_lag1']
    df['spy_direction_lag2'] = df['spy_direction_lag2']
    df.dropna(inplace=True)
    return df

def split_feature(df):
    features_list = ['d1', 'd2', 'd3', 'd4', 'd5', 'week_sin', 'week_cos', 'max_lag1', 'max_lag2', 'max_lag3', 'spy_direction_lag1', 'intraday_range_lag1', 'spy_direction_lag2']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list.extend(weekdays)
    X = df.loc[:, features_list]
    y = df.apply(lambda x: 1 if x['Max'] > 2 else 0, axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y)
    return X_train, X_test, y_train, y_test

# Create features and split data
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# Encode categorical features
le_features = LabelEncoder()
for column in ['d1', 'd2', 'd3', 'd4', 'd5']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = X_test[column].map(lambda s: le_features.transform([s])[0] if s in le_features.classes_ else -1)

# Train and tune the volatility model (RandomForestClassifier)
rf = RandomForestClassifier(random_state=12, class_weight='balanced')
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='recall', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Evaluate the volatility model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Feature Importance
feature_importances = model.feature_importances_
print("Feature Importances:", dict(zip(X_train.columns, feature_importances)))


direction_X = df[['intraday_range_lag1', 'spy_direction_lag2']].dropna()
direction_y = (df['spy_direction_lag1'] > 0).astype(int).loc[direction_X.index]
print("Unique classes in direction_y:", np.unique(direction_y))
direction_X_train, direction_X_test, direction_y_train, direction_y_test = train_test_split(direction_X, direction_y, test_size=0.2, random_state=14)
direction_model = LogisticRegression(random_state=12)
direction_model.fit(direction_X_train, direction_y_train)
direction_pred = direction_model.predict(direction_X_test)
directional_accuracy = (direction_pred == direction_y_test).mean()
print(f"Directional Accuracy: {directional_accuracy:.2f}")

# Cross-validation
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(direction_model, direction_X, direction_y, cv=5, scoring='accuracy')
print(f"Cross-validated Directional Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")




# Calculate EV
test_df = X_test.copy()
test_df['y_true'] = y_test
test_df['y_pred'] = y_pred
test_df['Max'] = df.loc[X_test.index, 'Max']
wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
losses = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
print(f"Avg Max on wins: {wins}, Trades: {test_df['y_pred'].sum()}, Losses: {losses}")

option_cost = 0.50
tp = len(test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])])
fp = losses
if tp > 0:
    avg_max_wins = wins
    tp_profit = (tp * 0.5 * (option_cost * avg_max_wins - option_cost)) + (tp * 0.5 * -option_cost)
    fp_loss = fp * -option_cost
    total_profit_test = tp_profit + fp_loss
    ev_per_trade = total_profit_test / (tp + fp)
    total_trades_full = (tp + fp) * (216 / len(y_test))
    ev_per_day = (total_profit_test * (216 / len(y_test))) / 216
    print(f"EV per trade (test set): ${ev_per_trade:.2f}")
    print(f"EV per day (full dataset): ${ev_per_day:.2f}")

    # Adjust EV with directional accuracy
    if directional_accuracy > 0.5:
        adjustment_factor = directional_accuracy / 0.5
        adjusted_ev_per_day = ev_per_day * adjustment_factor
        print(f"Adjusted EV per day with {directional_accuracy:.2f} directional accuracy: ${adjusted_ev_per_day:.2f}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        14
           1       0.66      0.90      0.76        30

    accuracy                           0.61        44
   macro avg       0.33      0.45      0.38        44
weighted avg       0.45      0.61      0.52        44

Feature Importances: {'d1': np.float64(0.041582981247814114), 'd2': np.float64(0.028299893457559056), 'd3': np.float64(0.026004870884608303), 'd4': np.float64(0.03726132437364158), 'd5': np.float64(0.0441887444954329), 'week_sin': np.float64(0.039856328717971), 'week_cos': np.float64(0.02380335093999103), 'max_lag1': np.float64(0.1083653798615608), 'max_lag2': np.float64(0.12170939136356054), 'max_lag3': np.float64(0.10567998995222357), 'spy_direction_lag1': np.float64(0.12680790

In [18]:
print(daily_summary['spy_direction_lag1'].describe())
print((daily_summary['spy_direction_lag1'] > 0).sum(), (daily_summary['spy_direction_lag1'] <= 0).sum())

count    223.000000
mean       0.204147
std        2.926161
min       -8.690000
25%       -1.695000
50%        0.230000
75%        1.995000
max        8.190000
Name: spy_direction_lag1, dtype: float64
121 102


In [21]:
daily_summary

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,close_equity,high_equity,low_equity,Max,weekday,spy_direction_lag1,intraday_range_lag1,spy_direction_lag2,day_classification,d1,d2,d3,d4,d5,weekday_encoded,spy_direction_sma_5
0,2023-01-03,1.000000,10.880000,385.290,380.8400,385.4000,377.8310,10.880000,Tuesday,NaN,NaN,NaN,very_high,None,None,None,None,None,3,NaN
1,2023-01-04,2.363636,1.419355,382.630,383.7400,385.8800,380.0000,2.363636,Wednesday,-4.450,7.5690,NaN,high,very_high,None,None,None,None,4,NaN
2,2023-01-05,1.411765,1.078755,379.710,379.4000,381.8400,378.7600,1.411765,Thursday,1.110,5.8800,-4.45,low,high,very_high,None,None,None,2,NaN
3,2023-01-06,13.225806,1.000000,380.210,388.0300,389.2500,379.4127,13.225806,Friday,-0.310,3.0800,1.11,very_high,low,high,very_high,None,None,0,NaN
4,2023-01-09,2.790698,1.705069,390.490,387.8657,393.7000,387.6700,2.790698,Monday,7.820,9.8373,-0.31,high,very_high,low,high,very_high,None,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,2023-11-22,1.069966,2.871795,456.140,455.0300,456.2200,453.8895,2.871795,Wednesday,0.320,2.1711,2.59,high,average,very_high,average,average,low,4,0.832
220,2023-11-24,1.346939,1.042038,454.930,455.0000,455.6000,454.7300,1.346939,Friday,-1.110,2.3305,0.32,low,high,average,very_high,average,average,0,0.702
221,2023-11-27,1.612903,1.019108,454.360,454.5000,455.4901,454.0799,1.612903,Monday,0.070,0.8700,-1.11,average,low,high,average,very_high,average,1,0.584
222,2023-11-28,4.509804,1.215000,453.605,454.8800,456.2700,453.5100,4.509804,Tuesday,0.140,1.4102,0.07,very_high,average,low,high,average,very_high,3,0.402


In [30]:
# Ensure Max is in daily_summary_clean
daily_summary_clean['Max'] = daily_summary_clean[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)

# Corrected Monte Carlo Simulation with consistent max_move
def monte_carlo_simulation(daily_summary, volatility_preds, adjustments, n_simulations=10000):
    ev_results = {}
    # Debug: Calculate average Max for trades
    max_moves = []
    for idx in daily_summary.index:
        if idx in volatility_preds.index and volatility_preds.loc[idx] == 1:
            max_moves.append(daily_summary.loc[idx, 'Max'])
    print(f"Average Max for trades: {np.mean(max_moves):.2f}")
    
    # Debug: Calculate average max_move for correct direction
    max_moves_correct = []
    for idx in daily_summary.index:
        if idx in volatility_preds.index and volatility_preds.loc[idx] == 1:
            actual_direction = 'long' if daily_summary.loc[idx, 'close_equity'] - daily_summary.loc[idx, 'equity_open'] > 0 else 'short'
            max_move = daily_summary.loc[idx, 'Call_Max_Daily'] if actual_direction == 'long' else daily_summary.loc[idx, 'Put_Max_Daily']
            max_moves_correct.append(max_move)
    print(f"Average max_move for correct direction: {np.mean(max_moves_correct):.2f}")
    
    total_trades = sum(1 for idx in daily_summary.index if idx in volatility_preds.index and volatility_preds.loc[idx] == 1)
    print(f"Total trades in Monte Carlo: {total_trades}")
    
    for adj in adjustments:
        total_profits = []
        correct_predictions = 0
        for _ in range(n_simulations):
            profit = 0
            for idx in daily_summary.index:
                if idx in volatility_preds.index and volatility_preds.loc[idx] == 1:
                    prev_day_direction = daily_summary.loc[idx, 'spy_direction_lag1']
                    trend_sma = daily_summary.loc[idx, 'spy_direction_sma_5']
                    prob_long = get_directional_prob(prev_day_direction, trend_sma, adj)
                    direction = np.random.choice(['long', 'short'], p=[prob_long, 1 - prob_long])
                    actual_direction = 'long' if daily_summary.loc[idx, 'close_equity'] - daily_summary.loc[idx, 'equity_open'] > 0 else 'short'
                    max_move = daily_summary.loc[idx, 'Max']  # Use Max for consistency
                    option_cost = 0.50
                    if direction == actual_direction:
                        profit += (option_cost * max_move - option_cost)
                        correct_predictions += 1
                    else:
                        profit -= option_cost
            total_profits.append(profit)
        avg_profit = np.mean(total_profits)
        ev_per_trade = avg_profit / total_trades if total_trades > 0 else 0
        directional_accuracy = correct_predictions / (total_trades * n_simulations)
        ev_results[adj] = ev_per_trade
        print(f"Adjustment {adj}: EV per trade = ${ev_per_trade:.2f}, Directional Accuracy = {directional_accuracy:.2%}")
    best_adjustment = max(ev_results, key=ev_results.get)
    print(f"Best adjustment: {best_adjustment} with EV per trade = ${ev_results[best_adjustment]:.2f}")
    return best_adjustment, ev_results

# Run Monte Carlo simulation with mean-reversal
adjustments = [-0.1, -0.05, 0, 0.05, 0.1]
volatility_preds = pd.Series(y_pred, index=X_test.index)
best_adjustment, ev_results = monte_carlo_simulation(daily_summary_clean, volatility_preds, adjustments)

# Apply best adjustment to calculate final EV
profit = 0
trades = 0
correct_predictions = 0
for idx in daily_summary_clean.index:
    if idx in volatility_preds.index and volatility_preds.loc[idx] == 1:
        trades += 1
        prev_day_direction = daily_summary_clean.loc[idx, 'spy_direction_lag1']
        trend_sma = daily_summary_clean.loc[idx, 'spy_direction_sma_5']
        prob_long = get_directional_prob(prev_day_direction, trend_sma, best_adjustment)
        direction = np.random.choice(['long', 'short'], p=[prob_long, 1 - prob_long])
        actual_direction = 'long' if daily_summary_clean.loc[idx, 'close_equity'] - daily_summary_clean.loc[idx, 'equity_open'] > 0 else 'short'
        max_move = daily_summary_clean.loc[idx, 'Max']
        option_cost = 0.50
        if direction == actual_direction:
            profit += (option_cost * max_move - option_cost)
            correct_predictions += 1
        else:
            profit -= option_cost
ev_per_trade = profit / trades if trades > 0 else 0
ev_per_day = (profit * (216 / len(y_test))) / 216
directional_accuracy = correct_predictions / trades
print(f"Final EV per trade with best adjustment {best_adjustment}: ${ev_per_trade:.2f}")
print(f"Final EV per day: ${ev_per_day:.2f}")
print(f"Final Directional Accuracy: {directional_accuracy:.2%}")

Average Max for trades: 4.81
Average max_move for correct direction: 4.66
Total trades in Monte Carlo: 41
Adjustment -0.1: EV per trade = $0.92, Directional Accuracy = 57.95%
Adjustment -0.05: EV per trade = $0.89, Directional Accuracy = 56.77%
Adjustment 0: EV per trade = $0.87, Directional Accuracy = 55.77%
Adjustment 0.05: EV per trade = $0.84, Directional Accuracy = 54.58%
Adjustment 0.1: EV per trade = $0.82, Directional Accuracy = 53.46%
Best adjustment: -0.1 with EV per trade = $0.92
Final EV per trade with best adjustment -0.1: $1.15
Final EV per day: $1.07
Final Directional Accuracy: 70.73%


In [23]:
# Calculate probability of green day following a green day
daily_summary_clean['current_direction'] = (daily_summary_clean['close_equity'] - daily_summary_clean['equity_open'] > 0).astype(int)
daily_summary_clean['prev_day_green'] = (daily_summary_clean['spy_direction_lag1'] > 0).astype(int)
prob_green_after_green = daily_summary_clean[daily_summary_clean['prev_day_green'] == 1]['current_direction'].mean()
prob_green_after_red = daily_summary_clean[daily_summary_clean['prev_day_green'] == 0]['current_direction'].mean()
print(f"Probability of green day after green day: {prob_green_after_green:.2f}")
print(f"Probability of green day after red day: {prob_green_after_red:.2f}")

Probability of green day after green day: 0.45
Probability of green day after red day: 0.75


In [28]:
max_moves = []
for idx in daily_summary_clean.index:
    if idx in volatility_preds.index and volatility_preds.loc[idx] == 1:
        actual_direction = 'long' if daily_summary_clean.loc[idx, 'close_equity'] - daily_summary_clean.loc[idx, 'equity_open'] > 0 else 'short'
        max_move = daily_summary_clean.loc[idx, 'Call_Max_Daily'] if actual_direction == 'long' else daily_summary_clean.loc[idx, 'Put_Max_Daily']
        max_moves.append(max_move)
print(f"Average max_move in Monte Carlo: {np.mean(max_moves):.2f}")

Average max_move in Monte Carlo: 4.66


In [25]:
daily_summary_clean.head(5)

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,close_equity,high_equity,low_equity,Max,weekday,spy_direction_lag1,intraday_range_lag1,spy_direction_lag2,day_classification,d1,d2,d3,d4,d5,weekday_encoded,spy_direction_sma_5,current_direction,prev_day_green
5,2023-01-10,1.480583,1.593750,388.42,390.63,390.650,386.27,1.593750,Tuesday,-2.6243,6.030,7.82,average,high,very_high,low,high,very_high,3,0.30914,1,0
13,2023-01-23,11.290323,1.190141,396.61,400.59,402.645,395.88,11.290323,Monday,6.1900,6.810,-0.49,very_high,very_high,average,very_high,average,very_high,1,0.43800,1,1
14,2023-01-24,1.569767,1.171946,398.70,400.17,401.150,397.64,1.569767,Tuesday,3.9800,6.765,6.19,average,very_high,very_high,average,very_high,average,3,0.34000,1,1
22,2023-02-03,2.920000,1.410042,413.50,412.36,416.970,411.09,2.920000,Friday,3.0000,5.430,5.51,high,very_high,high,high,high,high,0,2.38000,0,1
36,2023-02-24,1.601594,1.122807,394.80,396.39,397.250,393.64,1.601594,Friday,-0.4200,5.950,-1.49,average,very_high,average,very_high,low,high,0,-1.15000,1,0


In [7]:
daily_summary_filtered = daily_summary[daily_summary['Date'] > '2023-01-12']
daily_summary_filtered['Max'].describe()



# fig = px.histogram(daily_summary_filtered, x='day_classification', title='Distribution of Day Classification', histnorm='percent')
# fig.show()

count    216.000000
mean       3.964078
std        3.618854
min        1.172840
25%        1.831897
50%        2.654168
75%        5.063268
max       27.095238
Name: Max, dtype: float64

In [7]:
daily_summary['day_classification'].value_counts(normalize=True)

day_classification
high         0.370536
very_high    0.303571
average      0.232143
low          0.093750
Name: proportion, dtype: float64

In [ ]:
## Do not run. Old feature engineering code


from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 +1 )  # Week number in month
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)  # Assuming max 5 weeks
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['month_half'] = df['Date'].apply(lambda x: 1 if x.day <= 15 else 2)  # First or second half of the month
    df.dropna(inplace=True)
    return df
def split_feature(df):
    # X = df.loc[:,['d1', 'd2', 'd3', 'weekday_encoded', 'week_sin', 'week_cos']]  # Features
    y = df.loc[:,'day_classification']
    # features_list
    features_list = ['d1', 'week_sin']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    # features_list.extend(weekdays)
    X = df.loc[:,features_list]
    #
    # agggregate days in 2 groups: <2 and >=2

    y = y.apply(lambda x: 1 if 'high' in x else 0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y) # Stratify for class balance
    return X_train, X_test, y_train, y_test
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# Encode the features (d1, d2, d3)
le_features = LabelEncoder()
# for column in ['d1', 'd2', 'd3']:
for column in ['d1']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = le_features.transform(X_test[column])
# Train a classifier (RandomForest is a good starting point)
rf = RandomForestClassifier(random_state=12, class_weight='balanced') # You can try other models (e.g., Gradient Boosting)
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='balanced_accuracy', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model (convert predictions back to original labels for reporting)
#y_pred_labels = le.inverse_transform(y_pred)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate the model
#print(classification_report(y_test, y_pred))

# Feature Importance (to see which lags are most influential)
feature_importances = model.feature_importances_
print("Feature Importances:", feature_importances)


features = X_train.columns.tolist()


Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Classification Report:
               precision    recall  f1-score   support

           0       0.33      0.60      0.43        15
           1       0.67      0.40      0.50        30

    accuracy                           0.47        45
   macro avg       0.50      0.50      0.46        45
weighted avg       0.56      0.47      0.48        45

Feature Importances: [0.37604907 0.62395093]


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

very_high_range = 4
high_range = 2
average_range = 1.5

def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    df['d4'] = df.loc[:,'day_classification'].shift(4)
    df['d5'] = df.loc[:,'day_classification'].shift(5)
    df['spy_direction_lag1'] = (df['close_equity'] - df['equity_open']).shift(1)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 +1 )  # Week number in month
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)  # Assuming max 5 weeks
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['max_lag1'] = df['Max'].shift(1)
    df['max_lag2'] = df['Max'].shift(2)
    df['max_lag3'] = df['Max'].shift(3)
    df.dropna(inplace=True)
    return df

def split_feature(df):
    # X = df.loc[:,['d1', 'd2', 'd3', 'weekday_encoded', 'week_sin', 'week_cos']]  # Features
    # y = df.loc[:,'day_classification']
    # features_list
    features_list = ['d1', 'd2','d3', 'd4', 'd5', 'week_sin', 'week_cos', 'max_lag1', 'max_lag2', 'max_lag3','spy_direction_lag1']
    # bad features_list = ['d1', 'd2', 'd3', 'd4', 'd5', 'max_lag1', 'max_lag2', 'week_sin', 'week_cos']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list.extend(weekdays)
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list.extend(weekdays)
    X = df.loc[:,features_list]
    #
    # agggregate days in 2 groups: <2 and >=2

    # y = y.apply(lambda x: 1 if 'high' in x else 0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y) # Stratify for class balance
    return X_train, X_test, y_train, y_test

df = create_lags(daily_summary)

# Define y with Max > 3 before split_feature
y = df.apply(lambda x: 1 if x['Max'] > 2 else 0, axis=1)
X_train, X_test, y_train, y_test = split_feature(df)

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# Encode the features (d1, d2, d3)
le_features = LabelEncoder()
# for column in ['d1', 'd2', 'd3']:
for column in ['d1', 'd2', 'd3', 'd4', 'd5']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = X_test[column].map(lambda s: le_features.transform([s])[0] if s in le_features.classes_ else -1)
# Train a classifier (RandomForest is a good starting point)
rf = RandomForestClassifier(random_state=12, class_weight='balanced') # You can try other models (e.g., Gradient Boosting)
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='recall', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model (convert predictions back to original labels for reporting)
#y_pred_labels = le.inverse_transform(y_pred)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate the model
#print(classification_report(y_test, y_pred))

# Feature Importance (to see which lags are most influential)
feature_importances = model.feature_importances_
print("Feature Importances:", feature_importances)


features = X_train.columns.tolist()

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Classification Report:
               precision    recall  f1-score   support

           0       0.20      0.07      0.11        14
           1       0.67      0.87      0.75        30

    accuracy                           0.61        44
   macro avg       0.43      0.47      0.43        44
weighted avg       0.52      0.61      0.55        44

Feature Importances: [0.05103275 0.04223396 0.03720948 0.0531401  0.0525247  0.05162358
 0.03060912 0.13406068 0.14342153 0.12882946 0.14390956 0.02079943
 0.00756045 0.00748383 0.00912588 0.00941772 0.00782407 0.02291441
 0.00651361 0.00945958 0.01042943 0.01098009 0.00889658]


In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

very_high_range = 4
high_range = 2
average_range = 1.5

def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

# Assuming daily_summary is the result of your SQL query
# Add spy_direction_lag1
daily_summary['spy_direction_lag1'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(1)
daily_summary['Max'] = daily_summary[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    df['d4'] = df.loc[:,'day_classification'].shift(4)
    df['d5'] = df.loc[:,'day_classification'].shift(5)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 + 1)  # Week number in month
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)  # Assuming max 5 weeks
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['max_lag1'] = df['Max'].shift(1)
    df['max_lag2'] = df['Max'].shift(2)
    df['max_lag3'] = df['Max'].shift(3)
    df['spy_direction_lag1'] = df['spy_direction_lag1']
    df.dropna(inplace=True)
    return df

def split_feature(df):
    features_list = ['d1', 'd2', 'd3', 'd4', 'd5', 'week_sin', 'week_cos', 'max_lag1', 'max_lag2', 'max_lag3', 'spy_direction_lag1']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list.extend(weekdays)
    X = df.loc[:, features_list]
    y = df.apply(lambda x: 1 if x['Max'] > 2 else 0, axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y)
    return X_train, X_test, y_train, y_test

# Create features and split data
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# Encode categorical features
le_features = LabelEncoder()
for column in ['d1', 'd2', 'd3', 'd4', 'd5']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = X_test[column].map(lambda s: le_features.transform([s])[0] if s in le_features.classes_ else -1)

# Train and tune the model
rf = RandomForestClassifier(random_state=12, class_weight='balanced')
#grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='recall', verbose=2)
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='f1_weighted', verbose=2)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Evaluate the model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Feature Importance
feature_importances = model.feature_importances_
print("Feature Importances:", dict(zip(X_train.columns, feature_importances)))

# Calculate EV with directional uncertainty (50/50 chance)
y_pred = model.predict(X_test)
test_df = X_test.copy()
test_df['y_true'] = y_test
test_df['y_pred'] = y_pred
test_df['Max'] = df.loc[X_test.index, 'Max']
wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
losses = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
print(f"Avg Max on wins: {wins}, Trades: {test_df['y_pred'].sum()}, Losses: {losses}")

# EV calculation
option_cost = 0.50  # $0.50 per option
tp = len(test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])])
fp = losses
if tp > 0:
    avg_max_wins = wins
    # 50% chance of correct direction for true positives
    tp_profit = (tp * 0.5 * (option_cost * avg_max_wins - option_cost)) + (tp * 0.5 * -option_cost)
    fp_loss = fp * -option_cost
    total_profit_test = tp_profit + fp_loss
    ev_per_trade = total_profit_test / (tp + fp)
    total_trades_full = (tp + fp) * (216 / len(y_test))
    ev_per_day = (total_profit_test * (216 / len(y_test))) / 216
    print(f"EV per trade (test set): ${ev_per_trade:.2f}")
    print(f"EV per day (full dataset): ${ev_per_day:.2f}")
else:
    print("No true positives found for EV calculation.")

# Adjusted EV with commission
commission = 0.02
if tp > 0:
    tp_profit_adjusted = (tp * 0.5 * (option_cost * avg_max_wins - option_cost - commission)) + (tp * 0.5 * -(option_cost + commission))
    fp_loss_adjusted = fp * -(option_cost + commission)
    total_profit_test_adjusted = tp_profit_adjusted + fp_loss_adjusted
    ev_per_trade_adjusted = total_profit_test_adjusted / (tp + fp)
    ev_per_day_adjusted = (total_profit_test_adjusted * (216 / len(y_test))) / 216
    print(f"Adjusted EV per trade (with ${commission} commission): ${ev_per_trade_adjusted:.2f}")
    print(f"Adjusted EV per day (with ${commission} commission): ${ev_per_day_adjusted:.2f}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Classification Report:
               precision    recall  f1-score   support

           0       0.33      0.14      0.20        14
           1       0.68      0.87      0.76        30

    accuracy                           0.64        44
   macro avg       0.51      0.50      0.48        44
weighted avg       0.57      0.64      0.59        44

Feature Importances: {'d1': np.float64(0.05042571845535562), 'd2': np.float64(0.03773461457585133), 'd3': np.float64(0.03684348210087567), 'd4': np.float64(0.054779254012447744), 'd5': np.float64(0.05531544761875507), 'week_sin': np.float64(0.04923980709665994), 'week_cos': np.float64(0.026565145720925602), 'max_lag1': np.float64(0.13837189935770922), 'max_lag2': np.float64(0.15576669094081558), 'max_lag3': np.float64(0.13126276382692395), 'spy_direction_lag1': np.float64(0.15877

In [15]:
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='recall', verbose=2)
grid_search.fit(X_train, y_train)
model = grid_search.best_estimator_

y_pred_prob = model.predict_proba(X_test)[:, 1]
for threshold in [0.5, 0.55, 0.6]:
    y_pred = (y_pred_prob >= threshold).astype(int)
    print(f"\nClassification Report (Threshold {threshold}):\n", classification_report(y_test, y_pred))
    test_df['y_pred'] = y_pred
    wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
    losses = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
    print(f"Avg Max on wins: {wins}, Trades: {test_df['y_pred'].sum()}, Losses: {losses}")
    # Recalculate EV (same as before)

# Recalculate EV
test_df['y_pred'] = y_pred
wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
losses = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
print(f"Avg Max on wins: {wins}, Trades: {test_df['y_pred'].sum()}, Losses: {losses}")

# EV calculation
option_cost = 0.50
tp = len(test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])])
fp = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
if tp > 0:
    avg_max_wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
    tp_profit = (tp * 0.5 * (option_cost * avg_max_wins - option_cost)) + (tp * 0.5 * -option_cost)
    fp_loss = fp * -option_cost
    total_profit_test = tp_profit + fp_loss
    ev_per_trade = total_profit_test / (tp + fp)
    total_trades_full = (tp + fp) * (216 / len(y_test))
    ev_per_day = (total_profit_test * (216 / len(y_test))) / 216
    print(f"EV per trade (test set): ${ev_per_trade:.2f}")
    print(f"EV per day (full dataset): ${ev_per_day:.2f}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits

Classification Report (Threshold 0.5):
               precision    recall  f1-score   support

           0       0.33      0.14      0.20        14
           1       0.68      0.87      0.76        30

    accuracy                           0.64        44
   macro avg       0.51      0.50      0.48        44
weighted avg       0.57      0.64      0.59        44

Avg Max on wins: 6.578859721783396, Trades: 38, Losses: 12

Classification Report (Threshold 0.55):
               precision    recall  f1-score   support

           0       0.22      0.14      0.17        14
           1       0.66      0.77      0.71        30

    accuracy                           0.57        44
   macro avg       0.44      0.45      0.44        44
weighted avg       0.52      0.57      0.54        44

Avg Max on wins: 6.440922694233632, Trades: 35, Losses: 12

Classification Report (Threshold 0.6):
               precision    recall  f1-scor

In [11]:
dict(zip(features, feature_importances))

{'d1': np.float64(0.051032752002974),
 'd2': np.float64(0.042233956123144845),
 'd3': np.float64(0.037209476560659344),
 'd4': np.float64(0.05314010438774613),
 'd5': np.float64(0.05252469841133168),
 'week_sin': np.float64(0.0516235765568254),
 'week_cos': np.float64(0.030609120329670697),
 'max_lag1': np.float64(0.13406068376329644),
 'max_lag2': np.float64(0.1434215290009345),
 'max_lag3': np.float64(0.12882945969542645),
 'spy_direction_lag1': np.float64(0.14390956471352798),
 'weekday_encoded': np.float64(0.022914413391955032),
 'weekday_Friday': np.float64(0.006513613833280658),
 'weekday_Monday': np.float64(0.009459580138799077),
 'weekday_Thursday': np.float64(0.010429427031777226),
 'weekday_Tuesday': np.float64(0.010980090412637533),
 'weekday_Wednesday': np.float64(0.008896576265306547)}

In [16]:
print(len(daily_summary[daily_summary['Max'] >2]))
print(daily_summary[daily_summary['Max'] > 2]['Max'].describe())

151
count    151.000000
mean       5.127275
std        3.971972
min        2.015385
25%        2.602817
50%        3.548387
75%        6.075499
max       27.095238
Name: Max, dtype: float64


In [17]:
daily_summary['volume_lag1'] = daily_summary['volume_options'].shift(1)
df['volume_lag1'] = df['volume_lag1']
features_list.append('volume_lag1')

from sklearn.linear_model import LogisticRegression
direction_X = df[['spy_direction_lag1', 'volume_lag1']].dropna()
direction_y = (df['Max'] > 0).astype(int).loc[direction_X.index]  # Simplified direction (positive Max = call)
direction_X_train, direction_X_test, direction_y_train, direction_y_test = train_test_split(direction_X, direction_y, test_size=0.2, random_state=14)
direction_model = LogisticRegression(random_state=12)
direction_model.fit(direction_X_train, direction_y_train)
direction_pred = direction_model.predict(direction_X_test)
directional_accuracy = (direction_pred == direction_y_test).mean()
print(f"Directional Accuracy: {directional_accuracy:.2f}")

if directional_accuracy > 0.5:
    adjustment_factor = directional_accuracy / 0.5
    adjusted_ev_per_day = 0.54 * adjustment_factor
    print(f"Adjusted EV per day with {directional_accuracy:.2f} directional accuracy: ${adjusted_ev_per_day:.2f}")

KeyError: 'volume_options'

In [19]:
daily_summary

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,close_equity,Max,weekday,day_classification,d1,d2,d3,d4,d5,spy_direction_lag1,weekday_encoded
0,2023-01-03,1.000000,10.880000,385.290,380.8400,10.880000,Tuesday,very_high,None,None,None,None,None,NaN,3
1,2023-01-04,2.363636,1.419355,382.630,383.7400,2.363636,Wednesday,high,very_high,None,None,None,None,-4.450,4
2,2023-01-05,1.411765,1.078755,379.710,379.4000,1.411765,Thursday,low,high,very_high,None,None,None,1.110,2
3,2023-01-06,13.225806,1.000000,380.210,388.0300,13.225806,Friday,very_high,low,high,very_high,None,None,-0.310,0
4,2023-01-09,2.790698,1.705069,390.490,387.8657,2.790698,Monday,high,very_high,low,high,very_high,None,7.820,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,2023-11-22,1.069966,2.871795,456.140,455.0300,2.871795,Wednesday,high,average,very_high,average,average,low,0.320,4
220,2023-11-24,1.346939,1.042038,454.930,455.0000,1.346939,Friday,low,high,average,very_high,average,average,-1.110,0
221,2023-11-27,1.612903,1.019108,454.360,454.5000,1.612903,Monday,average,low,high,average,very_high,average,0.070,1
222,2023-11-28,4.509804,1.215000,453.605,454.8800,4.509804,Tuesday,very_high,average,low,high,average,very_high,0.140,3


In [16]:
import plotly.graph_objects as go

# Feature names and importances (based on your provided values and corrected order)
# features = ['d1', 'd2', 'd3', 'week_sin', 'week_cos', 'weekday_Friday', 'weekday_Monday', 
#             'weekday_Thursday', 'weekday_Tuesday', 'weekday_Wednesday']


importances = feature_importances
# Create a bar chart using Plotly
fig = go.Figure(data=[go.Bar(
    x=features,
    y=importances,
    marker_color='skyblue',  # Customize bar color
    text=importances,  # Display values on bars
    textposition='auto'  # Position text above bars
)])

# Customize the layout
fig.update_layout(
    title='Feature Importances in Random Forest Model',
    xaxis_title='Features',
    yaxis_title='Importance (Normalized)',
    xaxis={'tickangle': 45},  # Rotate x-axis labels for readability
    yaxis={'range': [0, 0.18]},  # Set y-axis range for better visibility
    bargap=0.2,  # Gap between bars
    plot_bgcolor='white',  # Background color
    showlegend=False  # No legend needed for a single bar chart
)

# Show the interactive plot
fig.show()


In [154]:
features

['d1',
 'd2',
 'd3',
 'week_sin',
 'week_cos',
 'weekday_Friday',
 'weekday_Monday',
 'weekday_Thursday',
 'weekday_Tuesday',
 'weekday_Wednesday']

In [157]:
fig = go.Figure(data=[go.Bar(
    y=features,
    x=importances,
    orientation='h',  # Horizontal bars
    marker_color='skyblue',
    text=importances,
    textposition='auto'
)])
fig.show()

In [181]:
from sklearn.inspection import permutation_importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
for i in range(len(features)):
    print(f"{features[i]}: {perm_importance.importances_mean[i]:.4f}")

d1: 0.1217
week_sin: 0.1652
weekday_encoded: 0.0783


In [17]:
import plotly.graph_objects as go

# Permutation importance values (from your report)

# Create a bar chart using Plotly
fig = go.Figure(data=[go.Bar(
    x=features,
    y=perm_importance.importances_mean,
    marker_color=[ 'green' if x >= 0 else 'red' for x in perm_importance.importances_mean],  # Green for positive, red for negative
    text=[f'{x:.4f}' for x in perm_importance.importances_mean],  # Display values on bars
    textposition='auto'  # Position text above bars
)])

# Customize the layout
fig.update_layout(
    title='Permutation Importances in Random Forest Model',
    xaxis_title='Features',
    yaxis_title='Permutation Importance (Change in Accuracy)',
    xaxis={'tickangle': 45},  # Rotate x-axis labels for readability
    yaxis={'range': [-0.07, 0.08]},  # Set y-axis range for better visibility
    bargap=0.2,  # Gap between bars
    plot_bgcolor='white',  # Background color
    showlegend=False  # No legend needed for a single bar chart
)

# Show the interactive plot
fig.show()

NameError: name 'perm_importance' is not defined

array([0.20416667, 0.09583333, 0.01666667, 0.0625    , 0.01666667,
       0.00416667])

In [6]:
dec = pd.read_pickle('0-2DTE_spy_options_01Dec23-31Dec23.pkl')
dec

,volume_options,volume_weighted_options,open_options,close_options,high_options,low_options,timestamp_options,number of trades_options,ticker,full_name,type,strike,expiry,equity_start_price,time_converted,open_equity,close_equity,high_equity,low_equity,timestamp_equity,number of trades_equity,_merge,equity_pct_change,options_pct_change,options_earliest_open
0,1,14.0200,14.02,14.02,14.02,14.02,1701441900000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 09:45:00,456.0443,456.1050,456.220,455.8100,1701441900000,18644,both,1.000000,1.0,14.02
1,3,14.9400,14.97,14.88,14.97,14.88,1701442800000,2,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:00:00,456.1100,456.2800,457.130,455.9499,1701442800000,41350,both,1.000144,1.06776,14.02
2,1,14.0500,14.05,14.05,14.05,14.05,1701443700000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:15:00,456.2650,455.6900,456.350,455.6000,1701443700000,23136,both,1.000484,1.00214,14.02
3,1,14.1900,14.19,14.19,14.19,14.19,1701445500000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:45:00,456.1800,456.3300,456.490,456.1200,1701445500000,14206,both,1.000298,1.012126,14.02
4,3,15.2633,15.25,15.29,15.29,15.25,1701446400000,2,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 11:00:00,456.3400,457.3400,457.410,455.1600,1701446400000,41644,both,1.000648,1.087732,14.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57629,1,0.0100,0.01,0.01,0.01,0.01,1703867400000,1,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 11:30:00,473.9400,474.1210,474.285,473.7700,1703867400000,24145,both,0.993845,1.0,0.01
57630,5,0.0100,0.01,0.01,0.01,0.01,1703873700000,1,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 13:15:00,474.8800,474.5000,474.910,474.3600,1703873700000,17916,both,0.995817,1.0,0.01
57631,2,0.0100,0.01,0.01,0.01,0.01,1703879100000,2,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 14:45:00,475.3450,475.5650,475.695,475.3200,1703879100000,17485,both,0.996792,1.0,0.01
57632,40,0.0100,0.01,0.01,0.01,0.01,1703881800000,2,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 15:30:00,475.7700,475.3117,475.780,475.3100,1703881800000,31027,both,0.997683,1.0,0.01


In [16]:
sum(dec['timestamp_equity'] == 1701441900000)

123